# Qa 05 model batch smoke test

Purpose: inspect the named geometry, dataset, or model diagnostic.

Prerequisites: install the project with the notebook extra and supply the research artifacts selected in the configuration cells. Launch Jupyter from the project root. See `docs/notebooks.md` for per-notebook inputs.

Outputs: displayed diagnostics and, where configured, exported figures/tables. Run cells from top to bottom. Saved outputs have been cleared.


# QA 05: Model Batch Smoke Test

Instantiate dataset, dataloader, and model, then run one forward pass and inspect the batch contract.

In [ ]:
from src.data_pipeline import build_split_dataloader, load_point_centric_arrays
from src.models import build_model_from_config
from notebooks.multisource_notebook_helpers import read_yaml

cfg = read_yaml("../configs/training.yaml")
arrays = load_point_centric_arrays(cfg["data"]["point_centric_dir"])
loader, ds = build_split_dataloader(arrays, cfg, split_name="train", shuffle=False)
sample = ds[0]
model = build_model_from_config(
    cfg,
    dynamic_input_dim=int(sample["x_dynamic"].shape[-1]) if "x_dynamic" in sample else 0,
    static_input_dim=int(sample["x_static"].shape[-1]),
    output_dim=4 if isinstance(sample["y"], dict) else int(sample["y"].shape[-1]),
    dynamic_feature_names=getattr(arrays, "dynamic_feature_names", None),
    source_dynamic_input_dim=int(sample["x_dynamic_sources"].shape[-1])
    if "x_dynamic_sources" in sample
    else None,
    source_geometry_input_dim=int(sample["source_geometry"].shape[-1])
    if "source_geometry" in sample
    else None,
    source_feature_names=getattr(arrays, "source_feature_names", None),
)
batch = next(iter(loader))
print(sorted(batch.keys()))

In [ ]:
# 1. Move the batch to device.
# 2. Run one forward pass.
# 3. Print output keys, tensor shapes, and loss inputs.